#### EDA with spark on compressed file

In [1]:
from pyspark.sql import SparkSession
from pyspark import SparkContext, SparkConf
from pyspark.sql import functions as F
import pyspark

## schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os

In [2]:
import pyspark
print(pyspark.__version__)

3.5.0


* install parquet library for pandas

In [3]:
# pip install pyspark

In [4]:
# pip install fastparquet

#### Create Spark context with `SparkConf`

In [5]:
spark= SparkSession.builder\
    .appName("readAWSData")\
    .getOrCreate()

In [6]:
cluster_url= 'local[*]'

# create config file
config = SparkConf()\
    .setAppName('readAWSCrawlData')\
    .setMaster(cluster_url)

## 2. Another way of creating Spark Session

=>
config= SparkConf()\
    .setAppName('readAWSCrawlData')\
    .setMaster('local[*]')


spark = SparkSession \
    .builder \
    .config(conf= config)\
    .getOrCreate()


### Read the compressed file 

### Reaad with spark session

In [7]:
config= SparkConf()\
    .setAppName('readAWSCrawlData')\
    .setMaster('local[*]')

spark = SparkSession \
    .builder \
    .config(conf= config)\
    .getOrCreate()

### Get downloaded gzip files from source

In [8]:
gz_data_path= "data"
gz_files_path= os.listdir(gz_data_path)
gz_files= [file_ for file_ in gz_files_path if file_.endswith(".gz")]
gz_files

['cdx-00015.gz']

In [9]:
# df_zipped= spark.read\
#     .format("csv")\
#     .option("compression", "gzip")\
#     .option("header", False)\
#     .load(f"../scripts/download_aws_crawl_data/data/gz_files[0]")

# df_zipped= spark.read.csv(
#     f"../scripts/download_aws_crawl_data/data/gz_files[0]",
#     f"gz_data_path + gz_files[0]",
#     header= True,
#     inferSchema= True
# )

## Spark detects gzip from file header
df_zipped= spark.read.csv(
    gz_data_path + '/' + gz_files[0],
    header= True,
    inferSchema= True
)

In [10]:
df_zipped.show(2)

+---+----------+-------------------------------------------------------------------------------------------------+--------------------+-----------------------------+----------------+---------------------------------------------+------------------+----------------------+----------------------------------------------------------------------------------------------------------------------------+-------------------+--------------------+
| ca|derivative|wiki)/audio_file_out_chop 20210124184737 {"url": "https://wiki.derivative.ca/Audio_File_Out_CHOP"| "mime": "text/html"| "mime-detected": "text/html"| "status": "200"| "digest": "6FCW7UKOBMV6BSJTIAO7OI5EOCDVLQW7"| "length": "13050"| "offset": "628841038"| "filename": "crawl-data/CC-MAIN-2021-04/segments/1610703550617.50/warc/CC-MAIN-20210124173052-20210124203052-00409.warc.gz"| "charset": "UTF-8"| "languages": "eng"}|
+---+----------+-------------------------------------------------------------------------------------------------+------------

In [11]:
df_zipped.printSchema()

root
 |-- ca: string (nullable = true)
 |-- derivative: string (nullable = true)
 |-- wiki)/audio_file_out_chop 20210124184737 {"url": "https://wiki.derivative.ca/Audio_File_Out_CHOP": string (nullable = true)
 |--  "mime": "text/html": string (nullable = true)
 |--  "mime-detected": "text/html": string (nullable = true)
 |--  "status": "200": string (nullable = true)
 |--  "digest": "6FCW7UKOBMV6BSJTIAO7OI5EOCDVLQW7": string (nullable = true)
 |--  "length": "13050": string (nullable = true)
 |--  "offset": "628841038": string (nullable = true)
 |--  "filename": "crawl-data/CC-MAIN-2021-04/segments/1610703550617.50/warc/CC-MAIN-20210124173052-20210124203052-00409.warc.gz": string (nullable = true)
 |--  "charset": "UTF-8": string (nullable = true)
 |--  "languages": "eng"}: string (nullable = true)



In [12]:
df_zipped.rdd.getNumPartitions()

1

## Repartition the `GZ` file into multiple `CSV` format

### Partitioning
Partitioning data in Apache Spark is a fundamental concept that plays a crucial role in the performance and efficiency of your Spark applications. Here’s a detailed explanation of what partitioning does and why it is important:

### What Partitioning Does

Partitioning in Spark refers to dividing a large dataset into smaller chunks, or partitions, which can be processed in parallel across the nodes of a cluster. Each partition is a logical division of the data that allows Spark to distribute and process the data concurrently.

### Importance of Partitioning

1. **Parallelism**:
   - **Improved Parallel Processing**: Partitioning enables parallel processing by allowing multiple tasks to work on different parts of the data simultaneously. This can significantly reduce the overall processing time.
   - **Optimal Resource Utilization**: By distributing tasks across available cores and nodes, Spark can better utilize the cluster's computational resources.

2. **Efficiency**:
   - **Data Locality**: Partitions can be processed on the node where the data resides, reducing the need for data transfer over the network and thus enhancing performance.
   - **Balanced Workload**: Proper partitioning ensures that the workload is evenly distributed across the cluster, preventing some nodes from being overloaded while others are idle.

3. **Memory Management**:
   - **Avoiding Out of Memory Errors**: Smaller partitions are easier to fit into memory, reducing the likelihood of out-of-memory errors and enabling more efficient garbage collection.

4. **Fault Tolerance**:
   - **Resilience to Failures**: If a node fails, only the tasks related to the partitions on that node need to be re-executed, rather than the entire job. This makes Spark applications more resilient to hardware failures.

5. **Scalability**:
   - **Handling Large Datasets**: Partitioning allows Spark to handle large datasets by breaking them into manageable chunks. This scalability is essential for big data processing.

### How Partitioning Works in Spark

- **Default Partitioning**: When you create an RDD or DataFrame, Spark automatically partitions it based on default settings or the source data's natural partitions (e.g., HDFS blocks).
- **Custom Partitioning**: You can explicitly control the number of partitions using operations like `repartition`, `coalesce`, and during data loading with options like `numPartitions`.

### Key Partitioning Operations

1. **repartition**:
   - **Description**: Increases or decreases the number of partitions and involves a full shuffle of the data, redistributing it across the nodes.
   - **Use Case**: When you need to increase parallelism or balance the data more evenly before a shuffle-intensive operation.

2. **coalesce**:
   - **Description**: Reduces the number of partitions without a full shuffle, typically by merging partitions within the same node.
   - **Use Case**: When you want to decrease the number of partitions for efficiency, often used after filtering operations to reduce the number of partitions.

3. **partitionBy** (for DataFrames):
   - **Description**: Specifies a column to partition the data by when writing it out to a file system.
   - **Use Case**: Optimizes data layout for future reads, especially for queries that filter by the partitioned column.


In [13]:
## Repartition and write out as CSV
df_zipped.repartition(20)\
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("data/repartition_data")

## Change the Column Names

In [ ]:
## shcema
df_zipped_ = df_zipped\
    .withColumn("c1_", F.col("_c0").cast(StringType())).drop("_c0")\
    .withColumn("c2_", F.col("_c1").cast(StringType())).drop("_c1")\
    .withColumn("url", F.col("_c2").cast(StringType())).drop("_c2")\
    .withColumn("mime", F.col("_c3").cast(StringType())).drop("_c3")\
    .withColumn("mime_type", F.col("_c4").cast(StringType())).drop("_c4")\
    .withColumn("status", F.col("_c5").cast(StringType())).drop("_c5")\
    .withColumn("digest_offset", F.col("_c6").cast(StringType())).drop("_c6")\
    .withColumn("length", F.col("_c7").cast(StringType())).drop("_c7")\
    .withColumn("offset", F.col("_c8").cast(StringType())).drop("_c8")\
    .withColumn("filename", F.col("_c9").cast(StringType())).drop("_c9")\
    .withColumn("content_charset", F.col("_c10").cast(StringType())).drop("_c10")\
    .withColumn("language", F.col("_c11").cast(StringType())).drop("_c11")


In [ ]:
df_zipped_.show(2)

### Partition
- To allow all the spark executors to process on sub-folders, rather handling one single Large file
- It is a lazy function

### NOTE: parition number
- For 4 Cores, starting with 16-32 paritions and adjusting based on the actual performance of `Spark Jobs`

In [ ]:
# df_zipped= df_zipped.repartition(4)

#### Write the partitioned file to dir.

In [ ]:
# %%time
# df_zipped.write.parquet("data_sources/aws_crawl_00005/")

In [ ]:
# df_zipped.rdd.getNumPartitions()

### Perform Substring of the URL column

#### Custom function
- user-defined function to split the column `url`

In [ ]:
## funciton to perform substring on Full URL path
def process_url(df):

    ## replace the NaN with NONE value
    # df= df.dropna()

    #df["url"]= df["url"].replace({ np.nan: 0})

    ## list of substrings
    ip_path= []
    resource_path= []
    time_stamp = []
    url= []

    
    # iterate over each row
    for j in range(0, len(df)):

#        if pd.notna(df["url"][j]):

        # split the url
        full_url= df["url"][j].split(' ')        
            
        # Get full path
        ip_resource= full_url[0].split(")")
        
        # get IP path
        ip_path.append(ip_resource[0])
        # Get Resource path
        resource_path.append(ip_resource[1])
    
        # get timestamp
        time_stamp.append(full_url[1])
        # Get the full URL
        url.append(full_url[3].replace('"', ''))

            
    df_new= df.copy()

    # Append the values
    df_new["Ip_path"]= ip_path
    df_new["Resource_path"]= resource_path
    df_new["TimeStamp"]= time_stamp
    df_new["URL"] = url

    ## drop URL column
    df_new= df_new.drop(columns= ["url"])

    return df_new

### Read the Parition the data

### Schema
- assign column names when reading file

In [ ]:
## Assign column name
new_col_names= ["c1_", "c2_", "url", "mime", "mime_type", "status", "digest_offset", "length", "offset", "filename",
                "content_charset", "language"]

# ## shcema
# schema_ = StructType([
#     StructField("c1_", StringType(), True),\
#     StructField("c2_", StringType(), True),\
#     StructField("url", StringType(), True), \
#     StructField("mime", StringType(), True), \
#     StructField("mime_type", StringType(), True), \
#     StructField("status", StringType(), True), \
#     StructField("digest_offset", StringType(), True), \
#     StructField("length", StringType(), True), \
#     StructField("offset", StringType(), True), \
#     StructField("filename", StringType(), True), \
#     StructField("content_charset", StringType(), True), \
#     StructField("language", StringType(), False)
# ])

In [ ]:
%%time
## infer schema for the column header and data type
## currently string type for all columns only
#    .schema(schema_)\

df_new= spark.read.format("parquet")\
    .load("data_sources/aws_crawl_00005/part-00000-4878556a-7ef5-4524-b862-7535a49b03be-c000.snappy.parquet")


In [ ]:
df_new= df_new\
    .withColumn("c1_", F.col("_c0").cast(StringType())).drop("_c0")\
    .withColumn("c2_", F.col("_c1").cast(StringType())).drop("_c1")\
    .withColumn("url", F.col("_c2").cast(StringType())).drop("_c2")\
    .withColumn("mime", F.col("_c3").cast(StringType())).drop("_c3")\
    .withColumn("mime_type", F.col("_c4").cast(StringType())).drop("_c4")\
    .withColumn("status", F.col("_c5").cast(StringType())).drop("_c5")\
    .withColumn("digest_offset", F.col("_c6").cast(StringType())).drop("_c6")\
    .withColumn("length", F.col("_c7").cast(StringType())).drop("_c7")\
    .withColumn("offset", F.col("_c8").cast(StringType())).drop("_c8")\
    .withColumn("filename", F.col("_c9").cast(StringType())).drop("_c9")\
    .withColumn("content_charset", F.col("_c10").cast(StringType())).drop("_c10")\
    .withColumn("language", F.col("_c11").cast(StringType())).drop("_c11")

In [ ]:
df_new.printSchema()

In [ ]:
df_new.show(2)

In [ ]:
df_new.count()

In [ ]:
df_new.rdd.getNumPartitions()

### User-Defined Function
- Register the `udf` to use with column 

In [ ]:
# First, Split by space
first_split= F.split(df_new["url"], " ")
df_new = df_new.withColumn("ip_and_resource", first_split.getItem(0))\
        .withColumn("time_stamp", first_split.getItem(1))\
        .withColumn("URL", first_split.getItem(3))

In [ ]:
df_new.printSchema()

In [ ]:
df_new.show(2)

#### Second split; on ip and resource

In [ ]:
split_col= F.split(df_new['ip_and_resource'], "\)")
df_new = df_new.withColumn('ip_path', split_col.getItem(0))\
            .withColumn('resource', split_col.getItem(1))

# convert the time stamp
#df_new= df_new.withColumn('DateTime', F.to_timestamp(F.col('time_stamp'), "yyyyMMddHHmmss")).show(2)

In [ ]:
## Drop the columns
#df_new= df_new.drop(*['ip_and_resource', 'time_stamp'])
df_new.show(2)

In [ ]:
df_new.rdd.getNumPartitions()

#### NOTE: Saves files in different folder as the data has been paritioned
- 

In [ ]:
# df_new.write.csv('data_sources/aws_crawl_00005/first_part_csv')

In [ ]:
## using coalase

df_new.repartition(1)\
    .write.format("data_sources/aws_crawl_00005/first_par_csv")\
    .option("header", "true")\
    .save("first_.csv")

In [ ]:
## Read the file

In [ ]:
%%time
df_parq= pd.read_csv("data_sources/aws_crawl_00005/first_part_csv/part-00004-9c5d555c-6feb-4672-a481-a67c0bf94d0a-c000.csv")